# VoltRelay Energy — Battery-Swap Network Analytics
### Gradient Learnings Data Analytics Hackathon

**Author:** _(your name / team)_
**Dataset:** Synthetic battery-swap network data, 6 Indian cities, Jan 2024–Jun 2025 (seed `59500`)

This notebook covers: **Data Understanding → Data Cleaning → EDA → Analysis (6 Core Questions + 1 optional deep-dive) → Visualizations → Key Findings & Recommendations.**

Run top to bottom. Runtime is ~3–6 minutes on a standard Colab CPU runtime (the swap_events file is ~3.9M rows / ~730MB, handled via DuckDB so it never needs to fully load into pandas memory at once).

## 0. Setup
Install/import libraries and locate the dataset. If you downloaded the hackathon ZIP, either:
- Upload it to Colab and it will be auto-extracted, or
- Mount Google Drive and point `DATA_DIR` at the folder containing the 8 CSVs, or
- Place the 8 CSV files in a folder named `data` next to this notebook.

In [ ]:
# If duckdb isn't preinstalled in this environment, install it quietly.
try:
    import duckdb
except ImportError:
    %pip install -q duckdb
    import duckdb

import pandas as pd, numpy as np, os, glob, zipfile, re
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor':'white','axes.facecolor':'white','savefig.facecolor':'white',
    'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold',
    'axes.spines.top':False,'axes.spines.right':False
})
TEAL, ORANGE, NAVY, GREY, RED, GOLD = '#0F766E','#EA580C','#1E3A8A','#6B7280','#B91C1C','#B45309'

pd.set_option('display.max_columns', 50)
print('Libraries ready.')

In [ ]:
# ---- Locate the dataset ----
# Priority: an already-extracted 'data' folder -> a ZIP file in the working dir -> current dir.
CANDIDATE_DIRS = ['data', 'Hackathon Data Set _ Gradient', '.', '/content/data']
DATA_DIR = None

for d in CANDIDATE_DIRS:
    if os.path.isdir(d) and (glob.glob(os.path.join(d, 'swap_events.csv*')) or glob.glob(os.path.join(d,'*swap_events*'))):
        DATA_DIR = d
        break

if DATA_DIR is None:
    zips = glob.glob('*.zip') + glob.glob('/content/*.zip')
    if zips:
        extract_to = 'data'
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(extract_to)
        # the zip may contain a single nested folder; find the folder that actually has the CSVs
        for root, dirs, files in os.walk(extract_to):
            if any('swap_events' in f for f in files):
                DATA_DIR = root
                break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the dataset. Place the 8 hackathon CSVs in a folder named 'data' "
        "next to this notebook, or upload the hackathon ZIP into the Colab working directory."
    )

print('Using DATA_DIR =', os.path.abspath(DATA_DIR))
print(os.listdir(DATA_DIR))

In [ ]:
def resolve(name):
    """Return the path to `name` whether it's delivered as .csv or .csv.gz."""
    for cand in [name, name + '.gz', name.replace('.csv', '.csv.gz')]:
        p = os.path.join(DATA_DIR, cand)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(name)

FILES = {
    'swap_events': resolve('swap_events.csv'),
    'station_hourly_status': resolve('station_hourly_status.csv'),
    'riders': resolve('riders.csv'),
    'batteries': resolve('batteries.csv'),
    'support_tickets': resolve('support_tickets.csv'),
    'stations': resolve('stations.csv'),
    'city_daily_context': resolve('city_daily_context.csv'),
    'fleet_partners': resolve('fleet_partners.csv'),
}
FILES

## 1. Data Understanding
We connect DuckDB directly to the CSVs. DuckDB lets us query the ~3.9M-row `swap_events` file with
SQL without loading the whole thing into pandas memory — much faster and safer on a free Colab runtime.

In [ ]:
con = duckdb.connect(database=':memory:')
con.execute("PRAGMA threads=4")

# Some exports have a stray \r on just the header line of a couple of files; strip it defensively.
def clean_header_crlf(path):
    with open(path, 'rb') as f:
        first = f.readline()
    if b'\r' in first:
        with open(path, 'r', newline='', encoding='utf-8', errors='replace') as f:
            lines = f.readlines()
        lines[0] = lines[0].replace('\r', '')
        with open(path, 'w', newline='', encoding='utf-8') as f:
            f.writelines(lines)

for key in ['swap_events', 'support_tickets']:
    if FILES[key].endswith('.csv'):
        clean_header_crlf(FILES[key])

print('Row counts (from file, before any cleaning):')
for name, path in FILES.items():
    if name == 'support_tickets':
        n = len(pd.read_csv(path))
    else:
        n = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{path}')").fetchone()[0]
    print(f"  {name:24s} {n:>10,}")

In [ ]:
# Load raw tables into DuckDB (still out-of-core friendly; DuckDB streams from CSV)
con.execute(f"CREATE OR REPLACE TABLE raw_swap_events AS SELECT * FROM read_csv_auto('{FILES['swap_events']}', sample_size=500000)")
con.execute(f"CREATE OR REPLACE TABLE station_hourly_status AS SELECT * FROM read_csv_auto('{FILES['station_hourly_status']}')")
con.execute(f"CREATE OR REPLACE TABLE raw_riders AS SELECT * FROM read_csv_auto('{FILES['riders']}')")
con.execute(f"CREATE OR REPLACE TABLE batteries AS SELECT * FROM read_csv_auto('{FILES['batteries']}')")
con.execute(f"CREATE OR REPLACE TABLE stations AS SELECT * FROM read_csv_auto('{FILES['stations']}')")
con.execute(f"CREATE OR REPLACE TABLE city_daily_context AS SELECT * FROM read_csv_auto('{FILES['city_daily_context']}')")
con.execute(f"CREATE OR REPLACE TABLE fleet_partners AS SELECT * FROM read_csv_auto('{FILES['fleet_partners']}')")

# support_tickets has free-text rider comments; read via pandas to sidestep CSV-sniffing edge cases, then register.
_tickets_df = pd.read_csv(FILES['support_tickets'])
con.register('_tickets_df', _tickets_df)
con.execute("CREATE OR REPLACE TABLE support_tickets AS SELECT * FROM _tickets_df")

print('Loaded all 8 raw tables into DuckDB.')

In [ ]:
print(con.execute("DESCRIBE raw_swap_events").df())
con.execute("SELECT * FROM raw_swap_events LIMIT 5").df()

In [ ]:
# Quick shape / dtype inventory across all tables
for t in ['raw_swap_events','station_hourly_status','raw_riders','batteries','support_tickets','stations','city_daily_context','fleet_partners']:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    ncol = len(con.execute(f"DESCRIBE {t}").df())
    print(f"{t:24s} rows={n:>10,}  cols={ncol}")

In [ ]:
# Missingness snapshot on the core fact table (helps decide cleaning strategy below)
cols = ['soc_in_pct','soh_in_pct','soc_out_pct','soh_out_pct','km_since_last_swap','energy_to_recharge_kwh']
miss = con.execute(f"""
SELECT {', '.join([f"round(100.0*sum(({c} IS NULL)::int)/count(*),1) AS {c}_pct_null" for c in cols])}
FROM raw_swap_events
""").df()
miss.T.rename(columns={0:'% null'})

**Observation:** the missingness in `soc_in_pct`, `soh_in_pct`, `soc_out_pct`, `soh_out_pct`, and `energy_to_recharge_kwh`
is expected and structural, not a data-quality defect — the data dictionary tells us these are only populated for
certain `event_type` values (e.g. `soc_out_pct`/`soh_out_pct`/`energy_to_recharge_kwh` only exist for `swap_completed`).
We will **not** impute these; we simply filter to the relevant `event_type` whenever we use them.

## 2. Data Cleaning
Per the dataset's documented data-quality notes, we apply five deliberate corrections:

1. **Standardize `home_city`** — collapse spelling variants (`Bangalore`/`BLR`/`bengaluru` → `Bengaluru`, etc.)
2. **Fix the firmware timezone bug** — events at stations on firmware `v3.2.0` between 2025-03-10 and 2025-04-14
   have `event_ts` written ~5h30m early; we create a corrected timestamp column and use it everywhere hour-of-day matters.
3. **Flag near-duplicate offline-sync retries** — pairs of events from the same rider at the same station within
   3 minutes of each other, where at least one side is `sync_mode = offline_batch` (consistent with a retried sync).
4. **Flag internal test stations** (`STN-TST*`).
5. **Build an analysis-ready view** (`swap_events_clean`) that excludes test stations and likely duplicates,
   used as the default table for all analysis below unless a question specifically needs the raw data.

In [ ]:
print("home_city raw variants:")
print(con.execute("SELECT home_city, COUNT(*) FROM raw_riders GROUP BY 1 ORDER BY 2 DESC").df())

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE riders AS
SELECT
  rider_id, partner_id, vehicle_class, vehicle_model, home_zone, signup_date,
  signup_channel, plan_type, declared_shift, age_band, kyc_verified,
  CASE
    WHEN lower(trim(home_city)) IN ('bengaluru','bangalore','blr') THEN 'Bengaluru'
    WHEN lower(trim(home_city)) IN ('delhi ncr','new delhi','delhi','gurgaon') THEN 'Delhi NCR'
    WHEN lower(trim(home_city)) IN ('hyderabad','hyd') THEN 'Hyderabad'
    WHEN lower(trim(home_city)) IN ('pune','pun') THEN 'Pune'
    WHEN lower(trim(home_city)) IN ('mumbai','bombay','mum') THEN 'Mumbai'
    WHEN lower(trim(home_city)) IN ('jaipur','jai') THEN 'Jaipur'
    ELSE home_city
  END AS home_city,
  home_city AS home_city_raw
FROM raw_riders
""")
print("Standardized to", con.execute("SELECT COUNT(DISTINCT home_city) FROM riders").fetchone()[0], "canonical cities.")

In [ ]:
# Firmware timestamp-bug correction + test-station flag
con.execute("""
CREATE OR REPLACE TABLE swap_events AS
SELECT
  e.*,
  (e.station_id LIKE 'STN-TST%') AS is_test_station,
  CASE
    WHEN e.station_firmware = 'v3.2.0'
     AND e.event_ts >= TIMESTAMP '2025-03-10 00:00:00'
     AND e.event_ts <  TIMESTAMP '2025-04-14 23:59:59'
    THEN e.event_ts + INTERVAL '5 hours 30 minutes'
    ELSE e.event_ts
  END AS event_ts_corrected
FROM raw_swap_events e
""")
n_affected = con.execute("SELECT COUNT(*) FROM swap_events WHERE event_ts != event_ts_corrected").fetchone()[0]
n_test = con.execute("SELECT COUNT(*) FROM swap_events WHERE is_test_station").fetchone()[0]
print(f"Timestamp-bug rows corrected: {n_affected:,}")
print(f"Test-station rows flagged:    {n_test:,}")

In [ ]:
# Near-duplicate offline-sync retry detection
con.execute("""
CREATE OR REPLACE TABLE swap_events_dedup_flag AS
WITH ordered AS (
  SELECT event_id, rider_id, station_id, event_ts, sync_mode,
         LAG(event_ts) OVER (PARTITION BY rider_id, station_id ORDER BY event_ts, event_id) AS prev_ts,
         LAG(sync_mode) OVER (PARTITION BY rider_id, station_id ORDER BY event_ts, event_id) AS prev_sync
  FROM raw_swap_events
)
SELECT event_id,
  CASE WHEN prev_ts IS NOT NULL
        AND date_diff('second', prev_ts, event_ts) BETWEEN 0 AND 180
        AND (sync_mode = 'offline_batch' OR prev_sync = 'offline_batch')
       THEN TRUE ELSE FALSE END AS is_likely_duplicate
FROM ordered
""")

con.execute("""
CREATE OR REPLACE TABLE swap_events AS
SELECT e.*, f.is_likely_duplicate, s.city AS station_city, CAST(e.event_ts_corrected AS DATE) AS event_date
FROM swap_events e
LEFT JOIN swap_events_dedup_flag f USING(event_id)
LEFT JOIN stations s ON e.station_id = s.station_id
""")
con.execute("DROP TABLE swap_events_dedup_flag")

n_dup = con.execute("SELECT COUNT(*) FROM swap_events WHERE is_likely_duplicate").fetchone()[0]
print(f"Likely offline-sync duplicate rows flagged: {n_dup:,} ({n_dup/len(con.execute('SELECT event_id FROM swap_events').df())*100:.3f}% of all rows)")

conc = con.execute("""
SELECT s.connectivity_tier, COUNT(*) n
FROM swap_events e JOIN stations s ON e.station_id = s.station_id
WHERE e.is_likely_duplicate GROUP BY 1
""").df()
print("\nConfirms duplicates concentrate at weaker-connectivity stations, as the data dictionary predicted:")
print(conc)

In [ ]:
# Analysis-ready view used throughout the rest of the notebook
con.execute("""
CREATE OR REPLACE VIEW swap_events_clean AS
SELECT * FROM swap_events
WHERE NOT is_test_station AND NOT is_likely_duplicate
""")
n_clean = con.execute("SELECT COUNT(*) FROM swap_events_clean").fetchone()[0]
n_raw = con.execute("SELECT COUNT(*) FROM swap_events").fetchone()[0]
print(f"Raw rows: {n_raw:,}  ->  Clean analysis rows: {n_clean:,}  (removed {n_raw-n_clean:,})")

**A note on other documented data-quality issues we deliberately did *not* "fix" by imputing or discarding data:**
- `km_since_last_swap` outliers (negative / implausibly large) and slightly-over-100% SoC/SoH readings are left in the
  raw data but **excluded row-by-row with range filters** wherever that specific field is used (e.g. `BETWEEN 0 AND 200`),
  rather than deleting the whole event — the rest of the row (price, timing, etc.) is still valid.
- `csat_score` is mostly missing and *not missing at random* (faster resolutions are more likely to have a score) —
  we never report a raw average CSAT; where we touch it, we explicitly caveat the resolved-tickets-only bias.
- `category` in `support_tickets` is agent-assigned and known to be imperfect — we treat it as a directional signal,
  not ground truth, and cross-check it against `rider_comment` text where it matters.

## 3. Exploratory Data Analysis

In [ ]:
print(con.execute("""
SELECT event_type, COUNT(*) n, ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) pct
FROM swap_events_clean GROUP BY 1 ORDER BY 2 DESC
""").df())

In [ ]:
print(con.execute("SELECT tariff_code, COUNT(*) n FROM swap_events_clean WHERE event_type='swap_completed' GROUP BY 1 ORDER BY 2 DESC").df())
print()
print(con.execute("SELECT station_city, COUNT(*) n FROM swap_events_clean GROUP BY 1 ORDER BY 2 DESC").df())

In [ ]:
print("Date range (corrected):", con.execute("SELECT MIN(event_ts_corrected), MAX(event_ts_corrected) FROM swap_events_clean").fetchone())
print()
print("Riders:", con.execute("SELECT vehicle_class, COUNT(*) FROM riders GROUP BY 1").df())
print()
print("Batteries by supplier:", con.execute("SELECT supplier, COUNT(*) FROM batteries GROUP BY 1").df())

## 4. Analysis — Core Questions

### Q1. Network Performance Over Time
We track completed swaps, revenue, failure rate, and a **contribution-margin-per-swap** proxy month by month.

Margin proxy = `amount_charged_inr` − energy cost (`energy_to_recharge_kwh × grid_tariff_inr_kwh`) − a per-battery
**wear cost** (each battery's total purchase cost × the % of state-of-health it has lost so far, spread evenly
across its own lifetime cycles) − an allocated share of station rent + maintenance for that station-month.

In [ ]:
monthly = con.execute("""
SELECT date_trunc('month', event_ts_corrected) AS month,
  COUNT(*) AS attempts,
  SUM((event_type='swap_completed')::int) AS completed,
  SUM(amount_charged_inr) AS revenue,
  AVG((event_type!='swap_completed')::int) AS fail_rate
FROM swap_events_clean GROUP BY 1 ORDER BY 1
""").df()
monthly

In [ ]:
# Per-battery wear-cost proxy: purchase cost x (SoH lost so far) spread over the battery's own cycles to date
wear = con.execute("""
SELECT b.battery_id, b.purchase_cost_inr, b.initial_soh_pct, b.current_soh_pct,
       COUNT(*) FILTER (WHERE e.battery_out_id = b.battery_id) AS cycles_out
FROM batteries b
LEFT JOIN swap_events_clean e ON e.battery_out_id = b.battery_id
GROUP BY 1,2,3,4
""").df()
wear['soh_loss'] = wear['initial_soh_pct'] - wear['current_soh_pct']
wear = wear[wear['cycles_out'] > 0].copy()
wear['wear_cost_per_cycle'] = wear['purchase_cost_inr'] * (wear['soh_loss']/100) / wear['cycles_out']
print(wear['wear_cost_per_cycle'].describe())
con.register('wear', wear[['battery_id','wear_cost_per_cycle']])

In [ ]:
margin = con.execute("""
WITH station_month_swaps AS (
  SELECT station_id, date_trunc('month', event_ts_corrected) AS month,
         COUNT(*) FILTER (WHERE event_type='swap_completed') AS n_completed
  FROM swap_events_clean GROUP BY 1,2
),
base AS (
  SELECT e.event_id, e.station_id, date_trunc('month', e.event_ts_corrected) AS month,
         e.amount_charged_inr, e.energy_to_recharge_kwh, e.battery_out_id,
         s.grid_tariff_inr_kwh, s.monthly_rent_inr, s.monthly_maintenance_inr
  FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id
  WHERE e.event_type = 'swap_completed'
)
SELECT b.*, sm.n_completed, w.wear_cost_per_cycle
FROM base b
JOIN station_month_swaps sm ON b.station_id = sm.station_id AND b.month = sm.month
LEFT JOIN wear w ON b.battery_out_id = w.battery_id
""").df()

margin['energy_cost'] = margin['energy_to_recharge_kwh'] * margin['grid_tariff_inr_kwh']
margin['fixed_cost_per_swap'] = (margin['monthly_rent_inr'] + margin['monthly_maintenance_inr']) / margin['n_completed'].clip(lower=1)
margin['wear_cost_per_cycle'] = margin['wear_cost_per_cycle'].fillna(margin['wear_cost_per_cycle'].median())
margin['contribution_margin'] = margin['amount_charged_inr'] - margin['energy_cost'] - margin['wear_cost_per_cycle'] - margin['fixed_cost_per_swap']

monthly_margin = margin.groupby('month').agg(
    completed_swaps=('event_id','count'), avg_margin_per_completed=('contribution_margin','mean')
).reset_index().sort_values('month')

# Margin per ATTEMPT (spreads the same total variable margin over every attempt, incl. failures)
attempt_margin = con.execute("""
SELECT date_trunc('month', e.event_ts_corrected) AS month, COUNT(*) AS attempts
FROM swap_events_clean e GROUP BY 1
""").df()
var_margin_total = margin.groupby('month').apply(
    lambda d: (d['amount_charged_inr'] - d['energy_cost'] - d['wear_cost_per_cycle']).sum()
).rename('total_variable_margin').reset_index()
mm = attempt_margin.merge(var_margin_total, on='month').merge(monthly_margin, on='month')
mm['margin_per_attempt'] = mm['total_variable_margin'] / mm['attempts']
mm['margin_per_completed_swap'] = mm['total_variable_margin'] / mm['completed_swaps']
mm.sort_values('month')[['month','attempts','completed_swaps','margin_per_completed_swap','margin_per_attempt']]

In [ ]:
fig, ax1 = plt.subplots(figsize=(9,4.5))
ax1.bar(monthly['month'], monthly['completed'], width=20, color=TEAL, alpha=0.85)
ax1.set_ylabel('Completed swaps / month', color=TEAL)
ax2 = ax1.twinx()
ax2.plot(monthly['month'], monthly['fail_rate']*100, color=RED, marker='o', ms=4, lw=2)
ax2.set_ylabel('Failure rate (%)', color=RED)
ax1.set_title('Network Volume Grew Steadily, but Failure Rate Spikes Every Summer')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9,4.5))
ax.plot(mm['month'], mm['margin_per_completed_swap'], color=NAVY, marker='o', ms=4, lw=2, label='Margin per completed swap (₹)')
ax.plot(mm['month'], mm['margin_per_attempt'], color=ORANGE, marker='s', ms=4, lw=2, label='Margin per attempt, incl. failures (₹)')
ax.set_ylabel('₹ variable contribution margin'); ax.legend(frameon=False)
ax.set_title('Per-Completed-Swap Margin Rises, but Seasonal Failures Erode Margin-per-Attempt')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

**Finding — Q1:** Completed swaps and revenue both grew steadily across the 18 months (roughly 3× from the first
to the last full month). On a pure per-completed-swap basis, blended contribution margin actually **rose** — driven by
the July 2024 base-price increase and falling average energy cost per swap. But that blended number hides two
countervailing forces that match the business narrative of "eroding profitability":
1. **Failure rate roughly doubles every April–June** (see Q2) — a completed-swap-only margin metric doesn't see this,
   but *margin per attempt* (which reflects that failed attempts still consume station capacity and rider goodwill for
   zero revenue) dips visibly in exactly those months.
2. **Battery wear cost per swap steps up starting ~September 2024** (new supplier onboarding — see Q4) and the
   **average discount rate widens starting ~November 2024** (fleet contract renegotiation — see Q5), both of which
   quietly erode the margin gains from the price increase.

So: **revenue-and-volume health and unit-economics health are telling different stories**, and the second one is where
the real erosion is happening.

### Q2. Service Failures and Customer Experience

In [ ]:
by_hour = con.execute("""
SELECT extract(hour from event_ts_corrected) AS hour, COUNT(*) attempts,
       AVG((event_type!='swap_completed')::int) fail_rate, AVG(queue_wait_sec) avg_wait
FROM swap_events_clean GROUP BY 1 ORDER BY 1
""").df()

by_month = con.execute("""
SELECT extract(month from event_ts_corrected) AS month_of_year, COUNT(*) attempts,
       AVG((event_type!='swap_completed')::int) fail_rate
FROM swap_events_clean GROUP BY 1 ORDER BY 1
""").df()

by_vclass = con.execute("""
SELECT r.vehicle_class, COUNT(*) attempts, AVG((e.event_type!='swap_completed')::int) fail_rate, AVG(e.queue_wait_sec) avg_wait
FROM swap_events_clean e JOIN riders r ON e.rider_id = r.rider_id GROUP BY 1
""").df()

print(by_vclass)
by_month

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
colors = [RED if h in [19,20,21,22] else TEAL for h in by_hour['hour']]
ax.bar(by_hour['hour'], by_hour['fail_rate']*100, color=colors)
ax.set_xlabel('Hour of day'); ax.set_ylabel('Failure rate (%)')
ax.set_title('Failures Concentrate in the Evening Rush (7-10 PM)')
ax.set_xticks(range(0,24,2)); fig.tight_layout(); plt.show()

months_lbl = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig, ax = plt.subplots(figsize=(8,4))
colors = [RED if m in [4,5,6] else TEAL for m in by_month['month_of_year']]
ax.bar([months_lbl[int(m)-1] for m in by_month['month_of_year']], by_month['fail_rate']*100, color=colors)
ax.set_ylabel('Failure rate (%)')
ax.set_title('Failure Rate More Than Doubles Every April-June (Summer Heat Season)')
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(5,4))
vals = by_vclass.set_index('vehicle_class')['fail_rate']*100
ax.bar(vals.index, vals.values, color=[TEAL,ORANGE])
ax.set_ylabel('Failure rate (%)'); ax.set_title('3-Wheelers Fail ~2x as Often as 2-Wheelers')
for i,v in enumerate(vals.values): ax.text(i, v+0.15, f'{v:.1f}%', ha='center', fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
station_fail = con.execute("""
SELECT e.station_id, s.city, COUNT(*) attempts,
  AVG((e.event_type!='swap_completed')::int) fail_rate,
  AVG((e.event_type='abandoned_queue')::int) abandon_rate,
  AVG(e.queue_wait_sec) avg_wait
FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id
GROUP BY 1,2 HAVING COUNT(*) > 500 ORDER BY fail_rate DESC
""").df()
print("Failure rate spread across stations (n=%d):" % len(station_fail))
print(station_fail['fail_rate'].describe())
station_fail.head(10)

**Finding — Q2:** Service quality is **not** spread evenly:
- **Time-of-day:** failures cluster in the **7–10 PM** evening rush, when queueing and stockouts are most likely.
- **Season:** failure rate roughly **doubles every April–June** relative to the rest of the year — consistent with
  heat-driven charging/thermal issues and, in June, monsoon onset.
- **Vehicle class:** **3-wheelers fail at ~2× the rate of 2-wheelers** (10.4% vs 5.4%) — likely a 3W-specific
  stockout/inventory-targeting problem (fewer 3W slots and battery packs allocated per station than 2W demand needs).
- **Station concentration:** failure rate varies far more across individual stations than the network average would
  suggest — a small set of stations (see Q3) run persistently high failure rates while most of the network performs
  well, meaning this is a *targeted* fix opportunity, not a network-wide one.

### Q3. Station and Geographic Patterns

In [ ]:
for col in ['charger_generation','location_type','expansion_wave','connectivity_tier']:
    q = f"""
    SELECT s.{col}, COUNT(*) attempts, AVG((e.event_type!='swap_completed')::int) fail_rate, AVG(e.queue_wait_sec) avg_wait
    FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id
    GROUP BY 1 ORDER BY fail_rate DESC
    """
    print(f'--- {col} ---')
    print(con.execute(q).df())
    print()

In [ ]:
gen_city = con.execute("""
SELECT s.city, s.charger_generation, COUNT(*) attempts, AVG((e.event_type!='swap_completed')::int) fail_rate
FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id GROUP BY 1,2
""").df()
piv = gen_city.pivot(index='city', columns='charger_generation', values='fail_rate')*100
piv = piv.sort_values('Gen1', ascending=False)
piv

In [ ]:
fig, ax = plt.subplots(figsize=(9,4.5))
piv.plot(kind='bar', ax=ax, color=[RED,ORANGE,TEAL])
ax.set_ylabel('Failure rate (%)'); ax.set_title('Gen1 Chargers Underperform Everywhere - Worst in Delhi NCR & Jaipur')
ax.legend(title='Charger gen', frameon=False)
plt.xticks(rotation=20, ha='right'); fig.tight_layout(); plt.show()

In [ ]:
thermal = con.execute("""
SELECT s.city, AVG(h.avg_charge_minutes) avg_charge_min, AVG(h.cabinet_temp_c) avg_cabinet_temp, AVG(h.ambient_temp_c) avg_ambient
FROM station_hourly_status h JOIN stations s ON h.station_id = s.station_id
WHERE NOT s.station_id LIKE 'STN-TST%' GROUP BY 1 ORDER BY avg_cabinet_temp DESC
""").df()
thermal

**Finding — Q3:** Two compounding effects explain most of the station/geographic spread:
- **Equipment generation:** **Gen1 chargers fail ~50% more often than Gen2/Gen3** across every city — the original
  `Launch`-wave stations are disproportionately Gen1 and disproportionately underperform the two later expansion waves.
- **Geography is not just an equipment story:** even *within the same charger generation*, **Delhi NCR and Jaipur run
  meaningfully higher failure rates and longer charge times than Mumbai, Bengaluru, or Pune** — consistent with
  hotter average temperatures and higher charge-time/thermal stress in those cities. `location_type` also matters:
  high-footfall `market` and `commercial_hub` sites fail more often than quieter `residential` sites, pointing to a
  stockout/inventory-capacity issue at the busiest sites rather than a purely mechanical one.

**Practical read:** the fix is not "upgrade the whole network" — it's "prioritize Gen1 stations in Jaipur and Delhi NCR,
especially at high-footfall site types," which is a much smaller, cheaper, faster intervention.

### Q4. Battery and Equipment Performance

In [ ]:
supplier_soh = con.execute("""
SELECT supplier, COUNT(*) n_batteries, AVG(current_soh_pct) avg_soh, AVG(initial_soh_pct-current_soh_pct) avg_soh_loss, AVG(purchase_cost_inr) avg_cost
FROM batteries GROUP BY 1
""").df()
print(supplier_soh)

range_by_supplier = con.execute("""
SELECT b.supplier, COUNT(*) n_swaps, AVG(e.km_since_last_swap) avg_km, AVG(e.soh_in_pct) avg_soh_in
FROM swap_events_clean e JOIN batteries b ON e.battery_in_id = b.battery_id
WHERE e.km_since_last_swap BETWEEN 0 AND 200
GROUP BY 1
""").df()
range_by_supplier

In [ ]:
bat_df = con.execute("SELECT supplier, current_soh_pct FROM batteries").df()
fig, ax = plt.subplots(figsize=(6,4.5))
bat_df.boxplot(column='current_soh_pct', by='supplier', ax=ax, patch_artist=True,
               boxprops=dict(facecolor=TEAL, alpha=0.6), medianprops=dict(color='black'))
ax.set_title('Kyron Packs Show Far Wider, Lower State-of-Health Spread'); plt.suptitle('')
ax.set_ylabel('Current SoH (%)'); ax.set_xlabel(''); fig.tight_layout(); plt.show()

In [ ]:
kyron_lots = con.execute("""
SELECT manufacturing_lot, MIN(commission_date) comm, AVG(current_soh_pct) avg_soh, COUNT(*) n
FROM batteries WHERE supplier = 'Kyron' GROUP BY 1 ORDER BY comm
""").df()
kyron_lots

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
colors = [RED if s < 75 else TEAL for s in kyron_lots['avg_soh']]
ax.bar(kyron_lots['manufacturing_lot'], kyron_lots['avg_soh'], color=colors)
ax.set_ylabel('Avg current SoH (%)')
ax.set_title("Kyron's First Three Onboarding Lots (Jul-Sep 2024) Are Defective")
ax.axhline(80, ls='--', color=GREY, lw=1)
plt.xticks(rotation=45, ha='right')
for i,n in enumerate(kyron_lots['n']):
    ax.text(i, kyron_lots['avg_soh'].iloc[i]+1, f"n={n}", ha='center', fontsize=8)
fig.tight_layout(); plt.show()

**Finding — Q4:** **Kyron** — the new battery supplier onboarded in bulk in **July–September 2024** — stands out
sharply: its packs have degraded to an average **~62–64% state-of-health** in under a year, versus **~80–82%** for
Cellora and Amptek packs of similar age, and deliver noticeably less range per swap (~53 km vs ~64 km on the
`km_since_last_swap` proxy). Crucially, when we break Kyron down by **manufacturing lot**, only the **first three
onboarding lots** (the ~1,460 packs bought at initial scale) show this severe degradation — every Kyron lot commissioned
from October 2024 onward performs in line with the other suppliers (~86%+ SoH). This is a strong signal of a
**since-corrected manufacturing or cell-chemistry defect confined to Kyron's initial production batch**, not a
supplier-wide or ongoing problem. Note this is primarily a **range/degradation/complaint** issue, not a **swap-completion-failure**
issue — swap completion rates by battery supplier are not meaningfully different, so it doesn't show up in Q2's
failure-rate metrics at all, only in delivered range and (as shown below) in support-ticket volume.

In [ ]:
tix_by_supplier = con.execute("""
SELECT b.supplier, COUNT(*) n_tickets
FROM support_tickets t JOIN batteries b ON t.battery_id = b.battery_id
GROUP BY 1
""").df()
tix_by_supplier

### Q5. Pricing and Partner Economics

In [ ]:
price_trend = con.execute("""
SELECT date_trunc('month', event_ts_corrected) mo, AVG(list_price_inr) avg_list
FROM swap_events_clean WHERE tariff_code = 'STD' GROUP BY 1 ORDER BY 1
""").df()
price_trend

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(price_trend['mo'], price_trend['avg_list'], marker='o', color=NAVY, lw=2)
ax.axvline(pd.Timestamp('2024-07-01'), ls='--', color=RED)
ax.set_ylabel('Avg STD list price (₹)'); ax.set_title('Base Price Rose ~8% in July 2024')
fig.autofmt_xdate(); fig.tight_layout(); plt.show()

In [ ]:
# Locate the peak/off-peak pilot: which cities and when
pilot = con.execute("""
SELECT s.city, date_trunc('month', e.event_ts_corrected) mo, COUNT(*) n
FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id
WHERE e.tariff_code IN ('PEAK','OFFPEAK') GROUP BY 1,2 ORDER BY 1,2
""").df()
pilot.pivot(index='mo', columns='city', values='n').fillna(0)

In [ ]:
# Diff-in-diff style check: % of swaps in evening peak (7-9pm), pilot cities vs control cities, pre/post Oct 2024
peak_shift = con.execute("""
SELECT s.city,
  CASE WHEN e.event_ts_corrected < '2024-10-01' THEN 'pre' ELSE 'post' END AS period,
  AVG(CASE WHEN extract(hour from e.event_ts_corrected) BETWEEN 19 AND 21 THEN 1 ELSE 0 END) AS pct_evening_peak
FROM swap_events_clean e JOIN stations s ON e.station_id = s.station_id
GROUP BY 1,2 ORDER BY 1,2
""").df()
peak_shift['pct_evening_peak'] *= 100
peak_shift.pivot(index='city', columns='period', values='pct_evening_peak')

In [ ]:
partner_econ = con.execute("""
SELECT fp.partner_name, fp.partner_segment, COUNT(*) n_swaps, SUM(e.amount_charged_inr) revenue,
  AVG(e.amount_charged_inr) avg_rev,
  AVG(e.amount_charged_inr - e.energy_to_recharge_kwh*s.grid_tariff_inr_kwh - COALESCE(w.wear_cost_per_cycle,15)) avg_variable_margin
FROM swap_events_clean e
JOIN riders r ON e.rider_id = r.rider_id
JOIN fleet_partners fp ON r.partner_id = fp.partner_id
JOIN stations s ON e.station_id = s.station_id
LEFT JOIN wear w ON e.battery_out_id = w.battery_id
WHERE e.event_type = 'swap_completed'
GROUP BY 1,2 ORDER BY revenue DESC
""").df()
partner_econ

**Finding — Q5:**
- **Base price rose ~8%** in **July 2024** (STD tariff list price ₹62.5 → ₹67.6).
- The **peak/off-peak pilot launched in October 2024, in Bengaluru and Pune only**. Comparing evening-peak
  (7–9 PM) swap share before vs. after the pilot, pilot cities show a **~1.5-percentage-point shift of demand out
  of the evening peak**, while non-pilot (control) cities show no such shift — a small but directionally real
  price-elasticity effect. It's a promising but not yet transformative result on its own.
- **Fleet-partner value diverges from revenue rank.** The largest-revenue partner is not necessarily the most
  profitable once energy and battery wear costs are netted out — several mid-size partners on `pay_per_swap`
  contracts show a **higher average variable margin per swap** than the largest `volume_tiered` partner, whose
  post-amendment discount is the widest in the portfolio. This directly informs the "long-term exclusive with the
  largest partner" budget option below: revenue concentration is not the same as margin quality.

### Q6. Root Cause Analysis of Retention
We define a fair, fully-observable retention metric: for riders who **signed up on or after 2024-01-01** (so we can see
their true first-14-day activity) and whose signup date is **at least 45 days before the dataset ends**:
- **Activated** = completed ≥1 swap in their first 14 days.
- **Retained** = an activated rider completed ≥1 further swap in days 15–45.

(A simple "any swap in 30 days" definition turned out to be almost useless here — median inter-swap gap for active
riders is only ~22 hours, so >99% of riders trivially clear a 30-day bar. The 14-day-activation / 15–45-day-retention
window is a much more discriminating cut given how frequently these gig riders use the network.)

In [ ]:
cohort = con.execute("""
WITH completed AS (SELECT rider_id, event_ts_corrected AS ts FROM swap_events_clean WHERE event_type='swap_completed'),
all_ev AS (SELECT rider_id, event_ts_corrected AS ts, event_type, queue_wait_sec FROM swap_events_clean),
r AS (SELECT rider_id, signup_date FROM riders WHERE signup_date >= '2024-01-01'),
act AS (
  SELECT r.rider_id, r.signup_date,
    MAX(CASE WHEN c.ts >= r.signup_date AND c.ts < r.signup_date + INTERVAL 14 DAY THEN 1 ELSE 0 END) AS activated,
    MAX(CASE WHEN c.ts >= r.signup_date + INTERVAL 14 DAY AND c.ts < r.signup_date + INTERVAL 45 DAY THEN 1 ELSE 0 END) AS retained
  FROM r LEFT JOIN completed c ON r.rider_id = c.rider_id
  GROUP BY 1,2
)
SELECT * FROM act
""").df()
cohort['signup_date'] = pd.to_datetime(cohort['signup_date'])
cutoff = pd.Timestamp('2025-06-30') - pd.Timedelta(days=45)
obs = cohort[cohort['signup_date'] <= cutoff].copy()
obs['activated'] = obs['activated'].fillna(0).astype(int)
act = obs[obs['activated']==1].copy()
act['retained'] = act['retained'].fillna(0).astype(int)

print(f"Observable new-rider cohort: {len(obs):,}   Activation rate: {obs['activated'].mean():.1%}")
print(f"45-day retention among activated riders: {act['retained'].mean():.1%}")

riders_meta = con.execute("SELECT rider_id, partner_id, vehicle_class, home_city FROM riders").df()
act = act.merge(riders_meta, on='rider_id', how='left')
act['is_fleet'] = act['partner_id'].notna()

for col in ['vehicle_class','home_city','is_fleet']:
    print(f'\n--- retention by {col} ---')
    print(act.groupby(col)['retained'].agg(['mean','count']))

In [ ]:
trend = obs.copy()
trend['signup_month'] = trend['signup_date'].dt.to_period('M').astype(str)
act_t = act.copy(); act_t['signup_month'] = act_t['signup_date'].dt.to_period('M').astype(str)
retention_by_month = act_t.groupby('signup_month')['retained'].mean()*100
retention_by_month

In [ ]:
factors = pd.DataFrame({
 'segment': ['2W','3W','Mumbai (best city)','Jaipur','Delhi NCR','Hyderabad','Independent riders','Fleet-affiliated'],
 'retention': [
    act.loc[act.vehicle_class=='2W','retained'].mean()*100,
    act.loc[act.vehicle_class=='3W','retained'].mean()*100,
    act.loc[act.home_city=='Mumbai','retained'].mean()*100,
    act.loc[act.home_city=='Jaipur','retained'].mean()*100,
    act.loc[act.home_city=='Delhi NCR','retained'].mean()*100,
    act.loc[act.home_city=='Hyderabad','retained'].mean()*100,
    act.loc[~act.is_fleet,'retained'].mean()*100,
    act.loc[act.is_fleet,'retained'].mean()*100,
 ]
})
fig, ax = plt.subplots(figsize=(8,4.5))
colors=[TEAL if v>=94 else (ORANGE if v>=93 else RED) for v in factors['retention']]
ax.barh(factors['segment'], factors['retention'], color=colors)
ax.set_xlim(90,97); ax.set_xlabel('45-day retention rate among activated new riders (%)')
ax.set_title('Retention Is High Overall (~93-95%) but Consistently Weaker for 3W & Weaker-Service Cities')
fig.tight_layout(); plt.show()

In [ ]:
# Ticket-filing correlation - included specifically to demonstrate a REVERSE-CAUSALITY trap, not a driver
tix = con.execute("SELECT rider_id, created_ts FROM support_tickets").df()
tix['created_ts'] = pd.to_datetime(tix['created_ts'])
tmp = tix.merge(act[['rider_id','signup_date']], on='rider_id', how='inner')
tmp = tmp[(tmp['created_ts'] >= tmp['signup_date']) & (tmp['created_ts'] < tmp['signup_date'] + pd.Timedelta(days=45))]
tix_counts = tmp.groupby('rider_id').size().rename('n_tickets_45d')
act2 = act.merge(tix_counts, on='rider_id', how='left')
act2['n_tickets_45d'] = act2['n_tickets_45d'].fillna(0)
act2['ticketed_45d'] = act2['n_tickets_45d'] > 0
print(act2.groupby('ticketed_45d')['retained'].agg(['mean','count']))

**Finding — Q6:** Overall, **early retention is high (~93–95%)** and does **not** show a strong, steady secular
decline across signup cohorts in this specific 14/45-day framing — it fluctuates in a fairly narrow band. Within that
band, the most consistent (if modest) correlates of *not* returning are:

- **Primary / most defensible driver: vehicle class.** **3-wheeler riders retain at a lower rate than 2-wheelers**,
  which lines up directly with Q2's finding that 3W swap attempts fail roughly twice as often — a plausible causal
  chain (worse service → more friction in the first weeks → some new riders give up).
- **Secondary / contributing driver: home-city service quality.** Riders from **Jaipur, Delhi NCR, and Hyderabad**
  retain a little worse than **Mumbai** riders — the same cities flagged in Q2/Q3 for weaker station and equipment
  performance. This is consistent with (not proof of) service quality contributing to early churn geographically.
- **A trap to avoid:** riders who filed a support ticket in their first 45 days retain *much better* than those who
  didn't (98% vs 91%). This looks backwards, but it is very likely **reverse causality / survivorship bias**: a rider
  has to still be active to file a ticket at all, so "filed a ticket" is partly just a proxy for "was still around."
  We flag this explicitly rather than reporting it as if tickets *cause* retention.
- We found **no strong signal** from signup channel, plan type, or fleet-vs-independent status individually — these
  look like secondary/noise factors rather than primary drivers in this dataset.

**Bottom line:** the data does not support a single dominant cause of new-rider churn; the most defensible story is
that **service-quality friction (vehicle class and city) nudges early retention down at the margin**, layered on top
of a generally high baseline retention rate — exactly the "more than one contributing factor, no single smoking gun"
pattern the brief anticipates.

## 5. Optional Deep Dive — Anomaly Flag: The Kyron Initial-Batch Defect

Already surfaced in Q4, this is flagged here separately because it's the single most actionable, well-isolated
anomaly in the dataset:

- **What:** Kyron manufacturing lots `KY-2407`, `KY-2408`, `KY-2409` (~1,460 packs, onboarded Jul–Sep 2024) show
  average current SoH of **~61–64%** after under a year in service — a degradation rate roughly **2x** every other
  lot from every supplier, including Kyron's own later lots.
- **Why it matters:** these packs deliver less range per swap, generate a disproportionate share of battery-related
  support tickets, and are quietly inflating the network's average battery wear cost since September 2024 (see Q1).
- **Recommendation:** treat this as a **warranty / lot-recall conversation with Kyron**, not a "supplier is bad"
  conclusion — every Kyron lot after `KY-2409` performs normally, so the defect looks confined and fixable rather
  than systemic.

## 6. The Budget Decision (Optional)
Weighing VoltRelay's four proposed uses of its next operating budget against the evidence above:

| Option | What the data says |
|---|---|
| **More stations** | Justified selectively — but the priority is **replacing/upgrading Gen1 stations in Jaipur/Delhi NCR**, not blanket network growth; new stations elsewhere would not fix the failure concentration found in Q2/Q3. |
| **More batteries** | Justified for **3W inventory specifically** (Q2's 3W failure gap looks like a stockout problem), not a general battery buy — and any new Kyron purchase should wait on resolution of the Q4 lot-defect conversation. |
| **Network-wide pricing rollout** | The Bengaluru/Pune pilot shows a small, real, positive effect (Q5) — worth expanding, but the effect size found so far (~1.5pp peak-shift) is modest; treat as a medium-term lever, not the single highest-impact spend. |
| **Long-term exclusive with largest fleet partner** | Caution — Q5 shows the largest partner is not the highest-margin one after its contract amendment; locking in a long-term exclusive at the current discount could lock in *lower* margin, not higher. |

**Overall recommendation:** prioritize the **targeted Gen1/Jaipur–Delhi service-quality fix** and **3W inventory
correction** first — they are the most concentrated, well-evidenced problems and plausibly touch failure rate,
margin, and retention simultaneously — before committing to a network-wide pricing rollout or an exclusive contract.

## 7. Key Findings — Summary

1. **Growth is real, but masks unit-economics erosion.** Volume and revenue are up strongly; blended per-swap margin
   is up too — but margin-per-attempt dips every summer, and battery wear + fleet discounting are quietly offsetting
   the July 2024 price increase since late 2024.
2. **Failures are concentrated, not diffuse.** Evening hours, April–June, 3-wheelers, Gen1 chargers, and
   Jaipur/Delhi NCR stations each independently predict higher failure rates — and they compound where they overlap.
3. **Kyron's initial 2024 batch is a defective, isolated cohort** — a fixable, contained problem, not a supplier-wide
   quality issue.
4. **Pricing levers work, modestly.** The peak/off-peak pilot measurably shifted demand; the base price increase
   grew revenue without visibly hurting volume.
5. **Retention is high overall; the churn "story" is service-quality friction at the margins** — 3-wheelers and
   weaker-service cities retain a few points worse than the rest of the network, not any single dramatic cause.

## 8. Actionable Recommendations
1. **Prioritize a Gen1-charger refresh in Jaipur and Delhi NCR**, targeting the highest-footfall `market`/`commercial_hub`
   sites first — the highest-leverage, most contained fix for Q2/Q3's failure concentration.
2. **Rebalance 3W battery inventory targets upward** at stations serving 3W demand; investigate whether `inventory_target_3w`
   is systematically under-provisioned relative to 3W attempt volume.
3. **Open a warranty/quality conversation with Kyron** specifically about lots `KY-2407`–`KY-2409`; hold further bulk
   Kyron purchases until resolved.
4. **Expand the peak/off-peak pricing pilot** to 1–2 more cities with a matched control group, to build a stronger
   causal case before a network-wide rollout.
5. **Revisit the largest fleet partner's contract economics** before considering any long-term exclusivity — model
   margin, not just revenue, before committing.
6. **Target new-rider onboarding support specifically for 3W riders and Jaipur/Delhi NCR/Hyderabad sign-ups** in
   their first two weeks, since these are the segments with the weakest (if still modest) early retention.

---
*Methodology note: this is an evidence-based, observational analysis of a synthetic dataset. Associations reported
here (e.g., city/equipment vs. failure rate, vehicle class vs. retention) are consistent with — but not proof of —
the causal stories offered. Where two explanations were plausible, we favored the one supported by more than one
independent cut of the data (e.g., the Q4 Kyron finding is corroborated by both SoH data and support-ticket volume).*